In [1]:
library(tidyverse)
library(oce)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.0     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.2     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.1     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
Lade nötiges Paket: gsw



In [73]:
ctd_ds <- read.csv("BCO-DMO/ctd.csv", na.strings="nd")

In [74]:
str(ctd_ds)

'data.frame':	288045 obs. of  21 variables:
 $ cruise_no : int  1 1 1 1 1 1 1 1 1 1 ...
 $ Cruise_ID1: chr  "93HG_001" "93HG_001" "93HG_001" "93HG_001" ...
 $ Cruise_ID2: chr  "CAR-001" "CAR-001" "CAR-001" "CAR-001" ...
 $ Year      : int  1995 1995 1995 1995 1995 1995 1995 1995 1995 1995 ...
 $ Month     : int  11 11 11 11 11 11 11 11 11 11 ...
 $ Day       : int  8 8 8 8 8 8 8 8 8 8 ...
 $ Date      : chr  "1995-11-08" "1995-11-08" "1995-11-08" "1995-11-08" ...
 $ Latitude  : num  10.5 10.5 10.5 10.5 10.5 10.5 10.5 10.5 10.5 10.5 ...
 $ Longitude : num  -64.7 -64.7 -64.7 -64.7 -64.7 ...
 $ press     : num  1.01 2.01 4.02 6.03 8.05 ...
 $ depth     : num  1 2 4 6 8 10 12 14 16 18 ...
 $ temp      : num  NA 27.5 27.5 27.5 27.5 ...
 $ sal       : num  36.6 36.6 36.6 36.6 36.6 ...
 $ potemp    : num  NA 27.5 27.5 27.5 27.5 ...
 $ sigma_t   : num  NA 23.8 23.8 23.8 23.8 ...
 $ sigma_0   : num  NA 23.8 23.8 23.8 23.8 ...
 $ O2_ml_L   : num  NA 3.96 3.98 3.98 3.99 ...
 $ beam_cp   : num  NA

In [75]:
ctd_ds$date <- paste(ctd_ds$Year,'-',ctd_ds$Month,'-',ctd_ds$Day, sep='')
ctd_ds$date <- as.Date(ctd_ds$date, format="%Y-%m-%d")

In [45]:
# import interpolation functions
source('interpolateData.r')

In [46]:
ctd_temp_int = interpolateDF(prepdataframe(ctd_ds, "temp"))

In [47]:
iso21_depth <- ctd_temp_int %>%
    group_by(date) %>%
    filter(depth > 6) %>%
    mutate(iso21 = value_int < 21) %>% # create new column that gives "True" for values at MLD
    filter(iso21 == T) %>% # only take "True" values 
    slice(1) # takes the first occurrence

In [48]:
iso21_df <- iso21_depth %>%
    rename(Isotherm_21 = depth) %>%
    select(date, Isotherm_21)

In [49]:
# MLD from sigma_t

In [50]:
ctd_sigma_t_int = interpolateDF(prepdataframe(ctd_ds, "sigma_t"))

In [51]:
ctd_sigma_t_diff <- ctd_sigma_t_int %>%
    group_by(date) %>%
    do(data.frame( sigma_t = .$value_int, sigma_t_diff = c(NA,diff(.$value_int)), depth = .$depth))

head(ctd_sigma_t_diff)

date,sigma_t,sigma_t_diff,depth
<date>,<dbl>,<dbl>,<dbl>
1995-11-08,23.76400,NA,0
1995-11-08,23.76400,0.000000000,1
1995-11-08,23.76400,0.000000000,2
1995-11-08,23.76950,0.005500000,3
1995-11-08,23.77500,0.005500000,4
1995-11-08,23.77895,0.003947205,5


In [52]:
# Criterion 1
mld_depth <- ctd_sigma_t_diff %>%
    group_by(date) %>%
    filter(depth > 9) %>%
    mutate(mld = sigma_t_diff >= 0.125 | sigma_t_diff <= -0.125) %>% # create new column that gives "True" for values at MLD
    filter(mld == T) %>% # only take "True" values 
    slice(1) # takes the first occurrence

In [53]:
#Criterion 2 -- better apparently (i think from some paper or email!)
mld_depth_2 <- ctd_sigma_t_diff %>%
    group_by(date) %>%
    filter(depth > 9) %>%
    mutate(mld = sigma_t >= sigma_t[1]+0.2 | sigma_t <= sigma_t[1]-0.2) %>% # create new column that gives "True" for values at MLD
    filter(mld == T) %>% # only take "True" values 
    slice(1) # takes the first occurrence

In [54]:
mld_df <- mld_depth_2 %>%
    rename(MLD = depth) %>%
    select(date, MLD)

In [55]:
# Calculate SST from Temperature at surface
sst_10m <- ctd_temp_int %>%
  group_by(date) %>%
  filter(depth <= 10) %>%
  summarize(sst = mean(value_int))

In [56]:
# Combine Isotherm and MLD data frames
CTD_combined_data <- list(iso21_df, mld_df, sst_10m) %>% 
  reduce(left_join, by = "date") %>% as.data.frame()

#write.csv(CTD_combined_data, "../DATA/January/CTD_Isotherm21_MLD.csv")

In [57]:
write.csv(CTD_combined_data, "processed/CTD_Isotherm21_MLD.csv")

In [58]:
saveRDS(CTD_combined_data, "processed/CTD_Isotherm21_MLD.rds")

# calculate upwelling index

In [39]:
UpwellingIndex <- ctd_temp_int %>%
    group_by(date) %>%
    filter(depth == 50) %>%
    mutate(ui = case_when(
                         value_int<=20.0 ~ "strong",
                         value_int<=21.0 ~ "moderate",
                         value_int<=22.0 ~ "weak",
                         value_int>22.0 ~ "relaxed"),
          upwelling = case_when(
                         value_int<=22.0 ~ "upwelling",
                         value_int>22.0 ~ "relaxed"),
          ) %>% select(date, ui, upwelling)

head(UpwellingIndex)

date,ui,upwelling
<date>,<chr>,<chr>
1995-11-08,relaxed,relaxed
1995-12-14,relaxed,relaxed
1996-01-13,relaxed,relaxed
1996-02-14,relaxed,relaxed
1996-03-13,moderate,upwelling
1996-04-16,moderate,upwelling


In [40]:
saveRDS(UpwellingIndex, "processed/CTD_UpwellingIndex.rds")

In [2]:
library(tidyverse)
library(oce)

ctd_ds <- read.csv("BCO-DMO/ctd.csv", na.strings = "nd")
ctd_ds$date <- paste(ctd_ds$Year, '-', ctd_ds$Month, '-', ctd_ds$Day, sep = '')
ctd_ds$date <- as.Date(ctd_ds$date, format = "%Y-%m-%d")

source('interpolateData.r')

# =============================================================================
# DIAGNOSTIC: CTD DATA COVERAGE
# =============================================================================

cat("=== RAW CTD DATA ===\n")
cat(sprintf("Total rows: %d\n", nrow(ctd_ds)))
cat(sprintf("Unique dates: %d\n", length(unique(ctd_ds$date))))
cat(sprintf("Date range: %s to %s\n", min(ctd_ds$date, na.rm = TRUE), max(ctd_ds$date, na.rm = TRUE)))

# Check temperature coverage at different depths
cat("\n=== TEMPERATURE DATA BY DEPTH ===\n")
temp_coverage <- ctd_ds %>%
  filter(!is.na(temp)) %>%
  mutate(depth_bin = cut(depth, breaks = c(0, 10, 25, 50, 75, 100, 200, Inf),
                         labels = c("0-10m", "10-25m", "25-50m", "50-75m", "75-100m", "100-200m", ">200m"))) %>%
  group_by(depth_bin) %>%
  summarize(
    n_obs = n(),
    n_dates = n_distinct(date),
    .groups = "drop"
  )
print(temp_coverage)

# Specifically check 50m (the classification depth)
cat("\n=== TEMPERATURE AT 50m (raw) ===\n")
temp_at_50_raw <- ctd_ds %>%
  filter(depth >= 48 & depth <= 52, !is.na(temp))
cat(sprintf("Observations within 48-52m: %d\n", nrow(temp_at_50_raw)))
cat(sprintf("Unique dates with T near 50m: %d\n", n_distinct(temp_at_50_raw$date)))

# Now check after interpolation
cat("\n=== AFTER INTERPOLATION ===\n")
ctd_temp_int <- interpolateDF(prepdataframe(ctd_ds, "temp"))
cat(sprintf("Interpolated rows: %d\n", nrow(ctd_temp_int)))
cat(sprintf("Unique dates after interpolation: %d\n", n_distinct(ctd_temp_int$date)))

# Check 50m specifically in interpolated data
temp_50m_int <- ctd_temp_int %>%
  filter(depth == 50)
cat(sprintf("Dates with interpolated T at exactly 50m: %d\n", nrow(temp_50m_int)))
cat(sprintf("Of those, non-NA values: %d\n", sum(!is.na(temp_50m_int$value_int))))

# Check what dates are MISSING from the 50m data
all_ctd_dates <- unique(ctd_ds$date)
dates_with_T50 <- unique(temp_50m_int$date[!is.na(temp_50m_int$value_int)])
missing_dates <- setdiff(all_ctd_dates, dates_with_T50)

cat(sprintf("\n=== DATES MISSING T(50m) ===\n"))
cat(sprintf("Total CTD dates: %d\n", length(all_ctd_dates)))
cat(sprintf("Dates with valid T(50m): %d\n", length(dates_with_T50)))
cat(sprintf("Missing T(50m): %d\n", length(missing_dates)))

# Why are they missing? Check max depth for missing dates
if (length(missing_dates) > 0) {
  cat("\n=== DIAGNOSING MISSING DATES ===\n")
  missing_diagnosis <- ctd_ds %>%
    filter(date %in% missing_dates) %>%
    group_by(date) %>%
    summarize(
      max_depth = max(depth, na.rm = TRUE),
      n_temp_obs = sum(!is.na(temp)),
      max_temp_depth = ifelse(any(!is.na(temp)), max(depth[!is.na(temp)]), NA),
      .groups = "drop"
    ) %>%
    mutate(
      reason = case_when(
        max_depth < 50 ~ "profile too shallow",
        is.na(max_temp_depth) ~ "no temp data",
        max_temp_depth < 50 ~ "temp data too shallow",
        TRUE ~ "interpolation issue"
      )
    )
  
  cat("Reasons for missing T(50m):\n")
  print(table(missing_diagnosis$reason))
  
  cat("\nSample of missing dates:\n")
  print(head(missing_diagnosis, 10))
}

# =============================================================================
# COMPARE TO UPWELLING CLASSIFICATION OUTPUT
# =============================================================================

cat("\n=== UPWELLING CLASSIFICATION ===\n")
upwelling_df <- ctd_temp_int %>%
  filter(depth == 50) %>%
  filter(!is.na(value_int)) %>%
  mutate(
    temp_50m = value_int,
    upwelling = case_when(
      value_int <= 22.0 ~ "upwelling",
      value_int > 22.0  ~ "relaxed"
    )
  )

cat(sprintf("Dates with upwelling classification: %d\n", nrow(upwelling_df)))
cat("\nUpwelling breakdown:\n")
print(table(upwelling_df$upwelling))

=== RAW CTD DATA ===
Total rows: 288045
Unique dates: 230
Date range: 1995-11-08 to 2017-01-17

=== TEMPERATURE DATA BY DEPTH ===
# A tibble: 7 × 3
  depth_bin  n_obs n_dates
  <fct>      <int>   <int>
1 0-10m       2126     230
2 10-25m      3399     230
3 25-50m      5644     230
4 50-75m      5646     229
5 75-100m     5515     229
6 100-200m   22660     229
7 >200m     242689     228

=== TEMPERATURE AT 50m (raw) ===
Observations within 48-52m: 989
Unique dates with T near 50m: 229

=== AFTER INTERPOLATION ===
Interpolated rows: 46230
Unique dates after interpolation: 230
Dates with interpolated T at exactly 50m: 230
Of those, non-NA values: 229

=== DATES MISSING T(50m) ===
Total CTD dates: 230
Dates with valid T(50m): 229
Missing T(50m): 1

=== DIAGNOSING MISSING DATES ===
Reasons for missing T(50m):

profile too shallow 
                  1 

Sample of missing dates:
# A tibble: 1 × 5
  date       max_depth n_temp_obs max_temp_depth reason             
  <date>         <dbl>    